# Control de calidad del MoCA — ReMePARK

Este cuaderno revisa la consistencia **interna** de la hoja `Hoja1` de una base con la estructura de `Remepark_MoCA_2526.xlsx`. No modifica el archivo original. Genera un Excel nuevo con incidencias por campo, cálculos independientes y copia de los datos de entrada.

## Guía rápida

1. Abre este archivo `.ipynb` en [Google Colab](https://colab.research.google.com/).
2. Ejecuta las celdas de arriba hacia abajo. Cuando se solicite, sube **un archivo `.xlsx`** con la misma estructura de columnas.
3. Al terminar, Colab descargará `QC_MoCA_ReMePARK.xlsx`.
4. Revisa primero **Incidencias**, utilizando `fila_excel` para localizar cada dato en el archivo original; compara después con **Resumen por fila**.

> **Alcance:** las reglas de puntaje y categorías aquí definidas reproducen el formato y las etiquetas presentes en la base recibida. Las categorías son variables de investigación; no sustituyen una evaluación clínica. Ajusta las reglas si cambia el formato del MoCA o el protocolo del estudio.

## 1. Cargar la base

La siguiente celda abre el selector de archivos de Colab, lee la hoja `Hoja1` y muestra tres registros para confirmar que cargó la tabla adecuada. La hoja `Hoja2` del archivo de referencia no contiene registros.

`reg.innn` se lee como texto para conservar su función de identificador. Si el archivo tiene otro nombre, no es necesario editar el código: usa el selector para subirlo. La celda espera exactamente un archivo.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Sube exactamente un archivo Excel.")
archivo = next(iter(uploaded))
df = pd.read_excel(archivo, sheet_name="Hoja1", dtype={"reg.innn": "string"})
print(f"Registros: {len(df)}; columnas: {len(df.columns)}")
display(df.head(3))

## 2. Definir los valores permitidos y la estructura del puntaje

| Componente | Columnas de origen | Puntaje máximo |
| --- | --- | ---: |
| Visuoespacial y ejecutivo | `MoCA.viso1` a `MoCA.viso5` | 5 |
| Denominación | `MoCA.den1` a `MoCA.den3` | 3 |
| Atención | `MoCA.attn1` (0–2), `MoCA.attn2` (0–1), `MoCA.attn3` (0–3) | 6 |
| Lenguaje | `MoCA.leng1` (0–2), `MoCA.leng2` (0–1) | 3 |
| Abstracción | `MoCA.abs` | 2 |
| Recuerdo diferido | `MoCA.rec.dif` | 5 |
| Orientación | `MoCA.orient` | 6 |

Cada ítem debe ser un **entero** entre cero y su máximo. Los cuatro subtotales se comparan con la suma de sus ítems; el total bruto se recalcula directamente a partir de todos los ítems (máximo 30), incluso si alguno de los subtotales guardados es incorrecto. `PT.educ` debe ser 0 o 1.

La celda comprueba además que existan las columnas necesarias. Si una falta, se detiene con una lista de los encabezados faltantes. Si tu base usa otros nombres o otra versión del instrumento, modifica `MAXIMOS` y `GRUPOS` antes de continuar.

In [ ]:
# Máximos por ítem de este formato (modifícalos si cambia el instrumento).
MAXIMOS = {
    **{f"MoCA.viso{i}": 1 for i in range(1, 6)},
    **{f"MoCA.den{i}": 1 for i in range(1, 4)},
    "MoCA.attn1": 2, "MoCA.attn2": 1, "MoCA.attn3": 3,
    "MoCA.leng1": 2, "MoCA.leng2": 1,
    "MoCA.abs": 2, "MoCA.rec.dif": 5, "MoCA.orient": 6,
    "PT.educ": 1,
}
GRUPOS = {
    "VISUO.TOT": [f"MoCA.viso{i}" for i in range(1, 6)],
    "DENOM.TOT": [f"MoCA.den{i}" for i in range(1, 4)],
    "ATTN.TOT": [f"MoCA.attn{i}" for i in range(1, 4)],
    "LENG.TOT": [f"MoCA.leng{i}" for i in range(1, 3)],
}
PARTES_TOTAL = list(GRUPOS) + ["MoCA.abs", "MoCA.rec.dif", "MoCA.orient"]
OTROS = ["MIS", "MoCA.TOTAL", "MoCA.TOTALcorre", "Dx.MoCA 26", "Dx.MoCA 24"]
REQUERIDAS = list(MAXIMOS) + list(GRUPOS) + OTROS
faltantes = sorted(set(REQUERIDAS) - set(df.columns))
if faltantes:
    raise ValueError(f"Faltan columnas requeridas: {faltantes}")
if df.columns.duplicated().any():
    raise ValueError("Hay nombres de columna duplicados; corrige el encabezado antes del QC.")

## 3. Ejecutar las comprobaciones

Para cada registro con datos del MoCA, el código verifica:

1. **Ítems:** presencia, rango e integridad (sin fracciones).
2. **Subtotales y total:** igualdad con las sumas recalculadas desde los ítems.
3. **Corrección educativa:** `MoCA.TOTALcorre = min(30, MoCA.TOTAL recalculado + PT.educ)`. Se comprueba el valor 0 o 1 de `PT.educ`, pero la base no incluye los años de escolaridad necesarios para verificar si correspondía otorgar ese punto.
4. **Clasificación a corte 26:** de 0 a 17, «Deterioro moderado a grave»; de 18 a 25, «Deterioro leve»; de 26 a 30, «Normal».
5. **Clasificación a corte 24:** de 0 a 17, «Deterioro moderado a grave»; de 18 a 23, «Deterioro leve»; de 24 a 30, «Normal».
6. **MIS:** puntaje entero de 0 a 15 y compatibilidad con las palabras recordadas libremente.

**Cómo se comprueba el MIS.** Con `r` palabras recordadas sin ayuda, cada una aporta 3 puntos; las otras `5 − r` podrían aportar entre 0 y 2 puntos cada una. Por tanto, debe cumplirse `3r ≤ MIS ≤ 3r + 2(5 − r)`, equivalente a `3r ≤ MIS ≤ 10 + r`. Por ejemplo, con `r = 4`, el MIS posible va de 12 a 14. La base no contiene evocación con pistas ni reconocimiento por palabra, así que **no permite recalcular el MIS exacto**. Una combinación dentro del intervalo es posible, pero no queda plenamente validada.

**Valores ausentes.** Una fila sin ningún campo MoCA informado recibe «Sin evaluación MoCA». En una fila parcialmente informada, los faltantes generan incidencias. Si algún ítem es inválido, los cálculos que dependen de él se marcan «No verificable»; el código no lo sustituye por cero. Las discrepancias se reportan para revisión, sin corregir automáticamente el dato original.

In [ ]:
# Cada incidencia conserva la fila original de Excel (encabezados en fila 1).
incidencias = []
def reportar(idx, campo, tipo, detalle, gravedad="Revisar"):
    incidencias.append({"fila_excel": idx + 2,
        "Consecutivo": df.iloc[idx].get("Consecutivo"),
        "reg.innn": df.iloc[idx].get("reg.innn"),
        "campo": campo, "tipo": tipo, "gravedad": gravedad, "detalle": detalle})

def numero(v):
    if pd.isna(v) or (isinstance(v, str) and not v.strip()): return None
    try: return float(v)
    except (TypeError, ValueError): return None

def valido(v, maximo):
    x = numero(v)
    return x is not None and np.isfinite(x) and x.is_integer() and 0 <= x <= maximo

def suma_valida(row, columnas, maximos):
    if not all(valido(row[c], maximos[c]) for c in columnas): return None
    return int(sum(numero(row[c]) for c in columnas))

def clase(p, corte):
    return "Deterioro moderado a grave" if p <= 17 else ("Deterioro leve" if p < corte else "Normal")

evaluacion = df[list(MAXIMOS) + list(GRUPOS) + OTROS].notna().any(axis=1)
calculados = []
for pos, (_, row) in enumerate(df.iterrows()):
    out = {"fila_excel": pos + 2, "Consecutivo": row.get("Consecutivo"), "reg.innn": row.get("reg.innn")}
    if not evaluacion.iloc[pos]:
        out["estado_evaluacion"] = "Sin evaluación MoCA"
        calculados.append(out)
        continue
    out["estado_evaluacion"] = "Con evaluación MoCA"
    for c, mx in MAXIMOS.items():
        if not valido(row[c], mx):
            tipo = "Faltante" if pd.isna(row[c]) else "Fuera de rango o no entero"
            reportar(pos, c, tipo, f"Esperado: entero de 0 a {mx}; observado: {row[c]!r}")
    for c, partes in GRUPOS.items():
        esperado = suma_valida(row, partes, MAXIMOS)
        out[f"calculado_{c}"] = esperado
        if esperado is None:
            reportar(pos, c, "No verificable", "Falta o es inválido al menos un ítem")
        elif not valido(row[c], sum(MAXIMOS[t] for t in partes)) or numero(row[c]) != esperado:
            reportar(pos, c, "Subtotal inconsistente", f"Registrado: {row[c]!r}; suma de ítems: {esperado}")

    # Calcular el total directamente desde ítems, sin confiar en subtotales guardados.
    items_total = [c for grupo in GRUPOS.values() for c in grupo] + ["MoCA.abs", "MoCA.rec.dif", "MoCA.orient"]
    bruto = suma_valida(row, items_total, MAXIMOS)
    out["calculado_MoCA.TOTAL"] = bruto
    if bruto is None:
        reportar(pos, "MoCA.TOTAL", "No verificable", "Falta o es inválido al menos un ítem del total")
    elif not valido(row["MoCA.TOTAL"], 30) or numero(row["MoCA.TOTAL"]) != bruto:
        reportar(pos, "MoCA.TOTAL", "Total inconsistente", f"Registrado: {row['MoCA.TOTAL']!r}; suma de ítems: {bruto}")

    corregido = min(30, bruto + int(numero(row["PT.educ"]))) if bruto is not None and valido(row["PT.educ"], 1) else None
    out["calculado_MoCA.TOTALcorre"] = corregido
    if corregido is None:
        reportar(pos, "MoCA.TOTALcorre", "No verificable", "Total bruto o PT.educ faltante/inválido")
    elif not valido(row["MoCA.TOTALcorre"], 30) or numero(row["MoCA.TOTALcorre"]) != corregido:
        reportar(pos, "MoCA.TOTALcorre", "Corrección inconsistente", f"Registrado: {row['MoCA.TOTALcorre']!r}; esperado: {corregido}")

    r, mis = row["MoCA.rec.dif"], row["MIS"]
    if pd.isna(mis):
        reportar(pos, "MIS", "Faltante", "MIS sin registro; no se reconstruye sin pistas y reconocimiento")
    elif not valido(mis, 15):
        reportar(pos, "MIS", "Fuera de rango o no entero", f"Esperado: entero de 0 a 15; observado: {mis!r}")
    elif valido(r, 5):
        inferior, superior = 3 * int(numero(r)), 10 + int(numero(r))
        if not inferior <= numero(mis) <= superior:
            reportar(pos, "MIS", "MIS incompatible con recuerdo libre", f"Recuerdo libre {r:g}: MIS posible {inferior}–{superior}; registrado {mis:g}")

    for corte in (26, 24):
        col = f"Dx.MoCA {corte}"
        esperado = clase(corregido, corte) if corregido is not None else None
        out[f"calculado_{col}"] = esperado
        if esperado is None:
            reportar(pos, col, "No verificable", "Falta total corregido calculable")
        elif pd.isna(row[col]) or str(row[col]).strip() != esperado:
            reportar(pos, col, "Clasificación inconsistente", f"Registrada: {row[col]!r}; esperada: {esperado}")
    calculados.append(out)

detalle = pd.DataFrame(incidencias, columns=["fila_excel", "Consecutivo", "reg.innn", "campo", "tipo", "gravedad", "detalle"])
comparacion = pd.DataFrame(calculados)
conteos = detalle.groupby("fila_excel").size() if not detalle.empty else pd.Series(dtype=int)
comparacion["n_incidencias"] = comparacion["fila_excel"].map(conteos).fillna(0).astype(int)
comparacion["estado_QC"] = np.where(comparacion.estado_evaluacion.eq("Sin evaluación MoCA"),
    "Sin evaluación MoCA", np.where(comparacion.n_incidencias.eq(0), "Sin incidencias", "Revisar"))
print("Resumen de filas:")
display(comparacion.estado_QC.value_counts().rename_axis("Estado").to_frame("Filas"))
print("Incidencias por tipo:")
display(detalle.tipo.value_counts().rename_axis("Tipo").to_frame("N") if not detalle.empty else "Sin incidencias")
display(detalle.head(25))

## 4. Descargar e interpretar el resultado

El archivo de salida contiene tres hojas:

| Hoja | Contenido |
| --- | --- |
| **Resumen por fila** | Identificadores, fila de Excel, subtotales y totales recalculados, clasificación esperada, número de incidencias y estado de control. |
| **Incidencias** | Una fila por hallazgo, con el campo afectado, motivo y detalle para cotejar con la fuente primaria. Una misma persona puede aparecer varias veces. |
| **Datos originales** | Copia sin correcciones de `Hoja1` para facilitar la comparación. |

`fila_excel` cuenta el encabezado como fila 1. **«Sin incidencias» indica que pasaron las verificaciones implementadas; no certifica la exactitud clínica de las respuestas originales ni la aplicación correcta del MoCA.** Revisa las incidencias con los formularios originales antes de cambiar la base.

In [ ]:
# Exportación: el origen y los resultados quedan en hojas separadas.
from io import BytesIO
salida = "QC_MoCA_ReMePARK.xlsx"
with pd.ExcelWriter(salida, engine="openpyxl") as writer:
    comparacion.to_excel(writer, sheet_name="Resumen por fila", index=False)
    detalle.to_excel(writer, sheet_name="Incidencias", index=False)
    df.to_excel(writer, sheet_name="Datos originales", index=False)
files.download(salida)